In [4]:
import numpy as np
import pathlib

# --- 请确认这是您的离线数据集目录 --- 
dataset_dir = '/home/jiangbo.chai/DATACENTER4/SWJTU_OOPAO/tutorials/scao_system/myAoSystem/RL4AO/agent/dreamerv3/logdir/AO1_R2DISC_lightRatio0.60/train_eps'

# 将其转换为 Path 对象
dataset_path = pathlib.Path(dataset_dir)

# 查找目录中的第一个 .npz 文件
try:
    sample_file = next(dataset_path.glob("*.npz"))
    print(f"成功找到样本文件: {sample_file.name}")
except StopIteration:
    print(f"错误：在目录 '{dataset_dir}' 中没有找到任何 .npz 文件。")
    sample_file = None

成功找到样本文件: 20250630T162009-ccdc002b98a14f65b0d53dcaacd42cd7-201.npz


In [5]:
# 加载文件并打印其中的所有 keys
if sample_file:
    data = np.load(sample_file, allow_pickle=True)
    
    print("\n文件中包含的 Keys:")
    keys = list(data.keys())
    print(keys)


文件中包含的 Keys:
['image', 'dmCoefs', 'wfsSingnal', 'is_terminal', 'is_first', 'reward', 'discount', 'action', 'logprob']


In [ ]:
# 打印每个 key 对应数据的形状 (shape)
if sample_file:
    print("\n每个 Key 对应的数据形状:")
    for key in keys:
        try:
            shape = data[key].shape
            print(f"  - {key}: {shape}")
        except AttributeError:
            print(f"  - {key}: (非数组对象，无shape属性)")

# --- 新增单元格：验证奖励和终止状态的索引 ---
if sample_file:
    # 选择一个中间的时间步 i 来进行检查，例如 i = 10
    i = 0
    print("\n" + "="*60)
    print(f"验证索引：检查从 state[{i}] 到 state[{i+1}] 的转移")
    print("="*60)

    # 1. 在状态 i (s_i) 采取的动作 a_i
    # 动作的形状通常是 (N, 1, Dims) 或 (N, Dims)，我们只关心第 i 个动作
    action_i = data['action'][i]
    print(f"在 state[{i}] 采取的动作 action[{i}]")

    # 2. 采取动作 a_i 后，在下一个时间步 i+1 观察到的奖励 r_{i+1}
    reward_i_plus_1 = data['reward'][i+1]
    print(f"转移后在 state[{i+1}] 获得的奖励 reward[{i+1}]: {reward_i_plus_1}")

    # 3. 采取动作 a_i 后，在下一个时间步 i+1 观察到的终止状态 d_{i+1}
    done_i_plus_1 = data['is_terminal'][i+1]
    print(f"转移后在 state[{i+1}] 的终止状态 is_terminal[{i+1}]: {done_i_plus_1}")

    print("\n--- 用于对比 ---")
    # 为了对比，我们打印一下 reward[i]
    # 这个奖励是在进入 state[i] 时获得的，是上一步动作 (action[i-1]) 的结果
    reward_i = data['reward'][i]
    print(f"进入 state[{i}] 时获得的奖励 reward[{i}]: {reward_i}")
    
    print("\n结论:")
    print(f"要构建 (s_i, a_i, r_{{i+1}}, s_{{i+1}}, d_{{i+1}}) 转移，")
    print(f"我们需要将 state[{i}]、action[{i}] 与它们导致的结果 reward[{i+1}] 和 is_terminal[{i+1}] 关联起来。")
    print("因此，在你的训练脚本循环中使用 `reward[i+1]` 和 `dones[i+1]` 是完全正确的。")


每个 Key 对应的数据形状:
  - image: (201, 120, 120, 3)
  - dmCoefs: (201, 1071)
  - wfsSingnal: (201, 1992)
  - is_terminal: (201,)
  - is_first: (201,)
  - reward: (201,)
  - discount: (201,)
  - action: (201, 78)
  - logprob: (201,)


IndexError: index 201 is out of bounds for axis 0 with size 201